# 03 — Train on Your Own Data

End-to-end walkthrough: load a CSV file, featurize, configure a model,
train, evaluate, and save a checkpoint ready for deployment.

**What you'll learn**
- `CageFusionDataModule.from_csv` — one-call featurization pipeline
- `CageFusionConfig` — set architecture, task type, and label names
- `Trainer` — train with early stopping and best-model checkpointing
- Save the checkpoint + scaler; load back with `CageFusionPipeline`

In [ ]:
%matplotlib inline

import pandas as pd
import torch

from cage_fusion import CageFusionConfig, AutoCageFusion
from cage_fusion.data import CageFusionDataModule
from cage_fusion.training import Trainer, TrainingArguments

## 1. Prepare your CSV

Your CSV must have:
- A `SMILES` column (or pass `smiles_col=` with the actual column name)
- One or more label columns (0/1 for classification, floats for regression)

Below we generate a tiny synthetic example.  Replace with your own file.

In [2]:
# ── Create a toy CSV ─────────────────────────────────────────────────────────
toy_data = [
    {"SMILES": "CC(=O)Oc1ccccc1C(=O)O",                         "active": 0, "toxic": 0},
    {"SMILES": "c1ccc2ccccc2c1",                                  "active": 0, "toxic": 0},
    {"SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",                   "active": 1, "toxic": 0},
    {"SMILES": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",             "active": 1, "toxic": 0},
    {"SMILES": "O=C(O)c1ccccc1O",                                  "active": 0, "toxic": 0},
    {"SMILES": "C1CCCCC1",                                         "active": 0, "toxic": 0},
    {"SMILES": "CCO",                                              "active": 0, "toxic": 0},
    {"SMILES": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                      "active": 1, "toxic": 0},
    {"SMILES": "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3","active": 1, "toxic": 1},
    {"SMILES": "O=C1c2ccccc2C(=O)c3ccccc13",                       "active": 0, "toxic": 1},
]

df = pd.DataFrame(toy_data)
df.to_csv("data/my_compounds.csv", index=False)

print(f"Dataset: {len(df)} rows")
print(df.head())

Dataset: 10 rows
                                SMILES  active  toxic
0                CC(=O)Oc1ccccc1C(=O)O       0      0
1                       c1ccc2ccccc2c1       0      0
2         CN1C=NC2=C1C(=O)N(C(=O)N2C)C       1      0
3  CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C       1      0
4                      O=C(O)c1ccccc1O       0      0


## 2. Build the data module

`from_csv` handles:
- Train / val (/ test) splitting
- RDKit auxiliary feature computation + scaler fitting
- ChemBERTa tokenization and embedding
- HDF5 streaming feature caching

In [3]:
LABEL_COLS = ["active", "toxic"]
CHECKPOINT_DIR = "data/tmp/cage_fusion_custom"

dm = CageFusionDataModule.from_csv(
    csv_path="data/my_compounds.csv",
    label_cols=LABEL_COLS,
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
    val_split=0.15,
    test_split=0.10,
    cache_dir="data/tmp/cage_features",
    batch_size=8,      # small for this toy example
)

print("Label names :", dm.label_names)
print("Train batches:", len(dm.train_loader))
print("Val batches  :", len(dm.val_loader))
print("Test batches :", len(dm.test_loader) if dm.test_loader else "None")

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

                  Featurisation parameters                  
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter  ┃ Value                                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ HDF5 path  │ data/tmp/cage_features/train_cage_fusion.h5 │
│ N samples  │ 7                                           │
│ seq_len    │ 512                                         │
│ embed_dim  │ 384                                         │
│ aux_dim    │ 217                                         │
│ num_labels │ 2                                           │
│ ids        │ False                                       │
│ batch_size │ 32                                          │
└────────────┴─────────────────────────────────────────────┘

INFO     Initialised HDF5 at data/tmp/cage_features/train_cage_fusion.h5 | N=7 emb=float32 ids=int32 aux_dim=217 labels=2


Featurising train: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.80it/s]

INFO     Normalising auxiliary features in data/tmp/cage_features/train_cage_fusion.h5 ...


...

INFO     Initialised HDF5 at data/tmp/cage_features/test_cage_fusion.h5 | N=1 emb=float32 ids=int32 aux_dim=217 labels=2


Featurising test: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.65it/s]

INFO     Normalising auxiliary features in data/tmp/cage_features/test_cage_fusion.h5 ...



Normalising test: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1844.46it/s]

INFO     Wrote 'auxiliary_features_normalized' to data/tmp/cage_features/test_cage_fusion.h5
INFO     [wk0/1] Dataset ready: N=1 graph_cache=objects emb_shape=(1, 512, 384)
Label names : ['active', 'toxic']
Train batches: 1
Val batches  : 1
Test batches : 1


## 3. Configure the model

In [4]:
config = CageFusionConfig(
    num_labels=len(dm.label_names),
    model_task="classification",
    label_names=dm.label_names,
    attn_mode="cross",          # paper's default — bidirectional gated co-attention
    use_fg_prompt=True,
    hidden_size=128,
)

print(config)

CageFusionConfig(num_labels=2, model_task='classification', attn_mode='cross', hidden_size=128, fusion_dim=901)


## 4. Build the model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoCageFusion.from_config(config).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

INFO     CAGEFusionModel initialised (attn_mode=cross)
Total parameters    : 7,350,800
Trainable parameters: 7,350,800


## 5. Train

In [6]:
args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    checkpoints_dir=CHECKPOINT_DIR,
    num_epochs=3,
    batch_size=8,
    learning_rate=3e-4,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

history = trainer.train()
print("Training complete.  Best val AUC:", max(history["val_auc"]))

INFO     Auto-built Adam optimizer  lr=3.00e-04  wd=0.00e+00  params=90
INFO     Training from epoch 1 to 3 | train batches: 1 | val batches: 1
INFO     Trainable params: 7,350,800


──────────────────────────────────────────────────── Epoch 1/3 ────────────────────────────────────────────────────

Train epoch 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.10it/s]

INFO     Top-5 attended FGs:
  1. pyrimidine                avg=1.0000
  2. carbonyl                  avg=0.5067
  3. phenyl                    avg=0.4418
  4. hydroxyl                  avg=0.4121
  5. carboxyl                  avg=0.2600


INFO     Epoch train | loss=0.5729 mcc=0.4061 auc=0.7000 pr=0.5417


...

Train Loss: 0.4224 (-0.0813)

      Validation Metrics       
┏━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric ┃  Value ┃         Δ ┃
┡━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━┩
│ Loss   │ 0.4776 │ (-0.0113) │
│ MCC    │ 0.0000 │ (+0.0000) │
│ AUC    │ 0.0000 │ (+0.0000) │
│ PR-AUC │    nan │     (nan) │
└────────┴────────┴───────────┘

                    Per-Task Validation Metrics                    
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Task      ┃         ROC-AUC ┃             MCC ┃          PR-AUC ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ active    │ 0.000 (+0.0000) │ 0.000 (+0.0000) │ 0.500 (+0.0000) │
│ toxic     │       nan (nan) │ 0.000 (+0.0000) │       nan (nan) │
├───────────┼─────────────────┼─────────────────┼─────────────────┤
│ Macro-Avg │          0.0000 │          0.0000 │             nan │
└───────────┴─────────────────┴─────────────────┴─────────────────┘

                     Learned Modality Scalers                      
┏━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Scaler      ┃   Value ┃ Avg. Rep Norm ┃ Scaled Norm ┃ Δ (Value) ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ scale_graph │ 10.0009 │        2.2225 │     22.2273 │ (+0.0003) │
│ scale_attn  │  0.9992 │       19.0614 │     19.0464 │ (-0.0003) │
│ scale_aux   │  0.5008 │       35.3300 │     17.6936 │ (+0.0003) │
└─────────────┴─────────┴───────────────┴─────────────┴───────────┘

INFO     Training complete.
INFO     Saved training history to data/tmp/cage_fusion_custom/training_history.csv
Training complete.  Best val AUC: 0.0


## 6. Plot training history

In [7]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs, history["val_auc"])
axes[1].set_title("Val ROC-AUC")

axes[2].plot(epochs, history["val_mcc"])
axes[2].set_title("Val MCC")

plt.tight_layout()
plt.show()

## 7. Save the scaler and config

In [8]:
import os

# The Trainer already saved best_model.pt to CHECKPOINT_DIR.
# Save the scaler (required by the pipeline) and the config.
dm.save_scaler(CHECKPOINT_DIR)
config.save_pretrained(CHECKPOINT_DIR)

print("Files saved:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    print(" ", f)

Files saved:
  auc_curve.png
  aux_features_scaler.pkl
  best_model.pt
  best_model_mcc.pt
  config.json
  latest_checkpoint.pt
  loss_curve.png
  mcc_curve.png
  pr_curve.png
  training_history.csv


## 8. Load back with the pipeline

In [9]:
from cage_fusion import CageFusionPipeline

pipe = CageFusionPipeline.from_pretrained(CHECKPOINT_DIR)

# Quick sanity check
print(pipe("CC(=O)Oc1ccccc1C(=O)O"))

INFO     CAGEFusionModel initialised (attn_mode=cross)


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

                 Featurisation parameters                 
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter  ┃ Value                                     ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ HDF5 path  │ /tmp/tmpq326qtzh/inference_cage_fusion.h5 │
│ N samples  │ 1                                         │
│ seq_len    │ 512                                       │
│ embed_dim  │ 384                                       │
│ aux_dim    │ 217                                       │
│ num_labels │ 0                                         │
│ ids        │ False                                     │
│ batch_size │ 256                                       │
└────────────┴───────────────────────────────────────────┘

INFO     Initialised HDF5 at /tmp/tmpq326qtzh/inference_cage_fusion.h5 | N=1 emb=float32 ids=int32 aux_dim=217 labels=0


Featurising inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.08it/s]

INFO     Normalising auxiliary features in /tmp/tmpq326qtzh/inference_cage_fusion.h5 ...



Normalising inference: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2084.64it/s]

INFO     Wrote 'auxiliary_features_normalized' to /tmp/tmpq326qtzh/inference_cage_fusion.h5
INFO     [wk0/1] Dataset ready: N=1 graph_cache=objects emb_shape=(1, 512, 384)
{'Original Index': 0, 'Id': None, 'SMILES': 'CC(=O)Oc1ccccc1C(=O)O', 'pred_class_active': 1, 'pred_class_toxic': 1, 'active': 0.2544049620628357, 'toxic': 0.1539982706308365, 'top_tokens': ''}


## 9. Push to HuggingFace Hub (optional)

`push_to_hub` uploads the three files the pipeline needs to reproduce inference:

| File | Contents |
|---|---|
| `best_model.pt` | Weights, config dict, best thresholds |
| `aux_features_scaler.pkl` | Fitted StandardScaler for aux features |
| `config.json` | Human-readable config |

Training-only artefacts (`latest_checkpoint.pt`, `best_model_mcc.pt`, PNGs, CSV) are skipped by default.
To reload from the Hub later just pass the repo id to `from_pretrained`:
```python
pipe = CageFusionPipeline.from_pretrained("your-username/cage-fusion-demo")
```

In [ ]:
# Replace 'your-username/cage-fusion-demo' with your actual HF repo id.
# Requires: pip install huggingface-hub

HF_REPO_ID = "your-username/cage-fusion-demo"   # ← edit this
HF_TOKEN   = None   # or pass token="hf_..." explicitly

# model= choices: "best" (best ROC-AUC), "best_mcc", "latest"
url = CageFusionPipeline.push_to_hub(
    CHECKPOINT_DIR,
    repo_id=HF_REPO_ID,
    model="best",    # ← swap to "best_mcc" or "latest" as needed
    token=HF_TOKEN,
    private=True,
)
print("Uploaded to:", url)

# --- reload from the Hub -------------------------------------------
# For "best_mcc" pass model_file_name="best_model_mcc.pt"
# For "latest"   pass model_file_name="latest_checkpoint.pt"
# pipe_hf = CageFusionPipeline.from_pretrained(
#     HF_REPO_ID, model_file_name="best_model.pt"
# )
